# 04. Activity Contributions

### Objective
Calculate per-window activity scores for Combat, Collection, and Exploration archetypes.

### I/O
- **Reads**: `../../data/processed/3_normalized_telemetry.csv`
- **Writes**: `../../data/processed/4_activity_contributions.csv`

In [1]:
import pandas as pd
import numpy as np

INPUT_PATH  = '../../data/processed/3_normalized_telemetry.csv'
OUTPUT_PATH = '../../data/processed/4_activity_contributions.csv'

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


### Activity score calculation - v2.2 feature groups.
All input columns are normalized [0,1] from notebook 03.

Feature group design choices (v2.2):

timeOutOfCombat is excluded from Exploration even though it sounds relevant. It accumulates passively for any player not in combat - including combat-intent players who just had no enemies nearby. It is also the arithmetic complement of timeInCombat, so including both would create inverse redundancy. Exploration is measured by active movement signals only: distanceTraveled and timeSprinting.

damage_per_hit and pickup_attempt_rate are v2.2 derived features computed in notebook 03. They capture sniper/heavy-weapon combat style and deliberate collection intent respectively - signals that raw hit count and pickup count alone miss.

Combat and Collection use mean() so their ceilings are 1.0. Exploration uses sum()/4 (half the collect ceiling) — this deliberately down-weights explore relative to collect so that a player who collects items but moves slightly does not get an explore score equal to their collect score, which was causing the Collection centroid to sit ambiguously between Collection and Exploration in pct space.


In [3]:
combat_features  = ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'damage_per_hit']
collect_features = ['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'pickup_attempt_rate']
explore_features = ['distanceTraveled', 'timeSprinting']

missing = [f for f in combat_features + collect_features + explore_features if f not in df.columns]
if missing:
    raise KeyError(f'Missing expected columns - run notebook 03 first: {missing}')

df['score_combat']  = df[combat_features].mean(axis=1)
df['score_collect'] = df[collect_features].mean(axis=1)
df['score_explore'] = df[explore_features].sum(axis=1) / 4  # 2 features / 4 — halves ceiling relative to collect for geometric separation
df['score_total']   = df['score_combat'] + df['score_collect'] + df['score_explore']

print('Calculated activity scores (v2.2 - averages, derived features, timeOutOfCombat excluded)')
print(f'score_combat  range: [{df["score_combat"].min():.3f}, {df["score_combat"].max():.3f}]')
print(f'score_collect range: [{df["score_collect"].min():.3f}, {df["score_collect"].max():.3f}]')
print(f'score_explore range: [{df["score_explore"].min():.3f}, {df["score_explore"].max():.3f}]')

Calculated activity scores (v2.2 - averages, derived features, timeOutOfCombat excluded)
score_combat  range: [0.000, 0.939]
score_collect range: [0.000, 0.750]
score_explore range: [0.000, 0.500]


In [4]:
# Convert raw scores to proportional percentages so each window sums to 1.0.
# Windows with no activity at all (score_total = 0) get equal distribution (1/3 each)
# rather than NaN, which would break downstream clustering.
df['pct_combat']  = df['score_combat']  / df['score_total'].replace(0, np.nan)
df['pct_collect'] = df['score_collect'] / df['score_total'].replace(0, np.nan)
df['pct_explore'] = df['score_explore'] / df['score_total'].replace(0, np.nan)

df[['pct_combat', 'pct_collect', 'pct_explore']] = (
    df[['pct_combat', 'pct_collect', 'pct_explore']].fillna(1/3)
)

print('Calculated activity percentages')

Calculated activity percentages


In [5]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')

Saved to ../../data/processed/4_activity_contributions.csv
